##  SETUP INSTRUCTIONS
-----
The following steps need to be followed before running the fastapi server and using temporary cloudflare deployment:
1. Click on the `folder icon` 📁 present in the `left bar` of the colab notebook.
2. Upload the files using the `upload option` icon
3. Upload the following files present in `backend folder` 📁 of the project:
- `main.py`
- `pronunciation.py`
- `serviceAccountKey.json`

In [ ]:
!pip install fastapi uvicorn pyngrok pymongo librosa transformers torch jiwer scipy firebase-admin kokoro soundfile agno groq kokoro soundfile python-dotenv speechbrain phonemizer jellyfish

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of spacy-curated-transformers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 133.1 MB/s eta 0:00:00
  

In [ ]:
# this was used only for testing purpose for secure handling use colab secrets
import os
os.environ["GROQ_API_KEY"]="<YOUR-GROQ-API-KEY>"
os.environ["MONGO_URI"] = "<YOUR-MONGODB-CLUSTER-LINK>"

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
!export ALLOW_ALL_ORIGINS=true
# then run your FastAPI uvicorn/gunicorn command

### After the below code displays message **"FASTAPI started"** wait for 30s for the server to load completely

In [ ]:
import subprocess, threading, time

def run_uvicorn():
    subprocess.run([
        "uvicorn", "main:app",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--timeout-keep-alive", "300"
    ])

threading.Thread(target=run_uvicorn, daemon=True).start()
time.sleep(5)  # wait for server to start
print("FastAPI started")

FastAPI started


In [ ]:
import re

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Wait for the public URL to appear in the output
print("Waiting for tunnel URL...")
for line in tunnel.stdout:
    line = line.decode()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print(f"\n✅ Public URL: {url}")
        print(f"\nSet this in your .env:\nVITE_API_URL={url}\n")
        break

Waiting for tunnel URL...

✅ Public URL: https://mandate-stations-candidate-mostly.trycloudflare.com

Set this in your .env:
VITE_API_URL=https://mandate-stations-candidate-mostly.trycloudflare.com



In [ ]:
# Used for testing if the fastapi server works fine or throws some error in development phase 
# in case your cloudflare link shows bad gateway or agent not working
!uvicorn main:app --host 0.0.0.0 --port 8000